# Unit 8 — Prefix Sums

A contest scoreboard stores a huge row of point values, then asks thousands of questions about totals between two positions.
Adding the same values again for every question is too slow.
This unit builds cumulative totals once, answers each 1D range in constant time, and extends the same idea to rectangular regions of a grid.

## Lesson 1 — 1D Prefix Sums

For an array `a` of length `n`, make a prefix array `pre` of length `n + 1`.
The extra first entry is `pre[0] = 0`, meaning that zero values have total zero.
For every `i` from `1` through `n`, use `pre[i] = pre[i - 1] + a[i - 1]`.
Therefore, `pre[i]` is the sum of the first `i` values of `a`, not the sum through array index `i`.

## Hand Trace the Extra Zero

For `a = [4, 1, 3, 2]`, the prefix array is `pre = [0, 4, 5, 8, 10]`.
To total the inclusive range from zero-indexed `l` through `r`, subtract the total before `l` from the total through `r`: `pre[r + 1] - pre[l]`.
For `l = 1` and `r = 3`, this is `pre[4] - pre[1] = 10 - 4 = 6`.
The `+ 1` is essential: using `pre[r]` drops `a[r]`, while subtracting `pre[l + 1]` also drops `a[l]`.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    query_count = int(tokens[1])
    values = []
    value_index = 0
    while value_index < n:
        values.append(int(tokens[2 + value_index]))
        value_index = value_index + 1

    pre = []
    pre.append(0)
    value_index = 0
    while value_index < n:
        pre.append(pre[value_index] + values[value_index])
        value_index = value_index + 1

    answer_text = ""
    token_position = 2 + n
    query_index = 0
    while query_index < query_count:
        left = int(tokens[token_position])
        right = int(tokens[token_position + 1])
        range_total = pre[right + 1] - pre[left]
        if query_index > 0:
            answer_text = answer_text + "\n"
        answer_text = answer_text + str(range_total)
        token_position = token_position + 2
        query_index = query_index + 1
    return answer_text

assert solve("7 3 4 1 3 2 8 5 9 0 0 1 4 4 6") == "4\n14\n22"

## Build Once, Query Many Times

Building `pre` visits each of the `n` values once, so it takes O(n) time.
Every inclusive range query then uses two prefix entries and one subtraction, so each query takes O(1) time.
Test a single-element range where `l == r`, a full-array range from `0` through `n - 1`, and a range ending at `n - 1`.
Those checks expose most mistakes involving `pre[0]`, `r + 1`, or the final array value.

## Lesson 2 — 2D Prefix Sums

Now imagine a contest map that asks for the point total inside many rectangular regions.
Make a prefix grid with one extra zero row and one extra zero column.
For a grid cell `(r, c)`, store its value plus the totals above and left, then subtract the upper-left overlap that was added twice.
The build rule is `pre[r + 1][c + 1] = grid[r][c] + pre[r][c + 1] + pre[r + 1][c] - pre[r][c]`.

## Hand Trace a Prefix Grid

For the grid `[[2, 1, 4], [3, 5, 6]]`, including the extra borders gives these prefix rows: `[0, 0, 0, 0]`, `[0, 2, 3, 7]`, and `[0, 5, 11, 21]`.
The rectangle from `(r1, c1) = (0, 1)` through `(r2, c2) = (1, 2)` contains `1, 4, 5, 6`.
Its total is `pre[r2 + 1][c2 + 1] - pre[r1][c2 + 1] - pre[r2 + 1][c1] + pre[r1][c1]`.
Here that is `pre[2][3] - pre[0][3] - pre[2][1] + pre[0][1] = 21 - 0 - 5 + 0 = 16`.

## Why Inclusion-Exclusion Works

Start with the prefix rectangle through `(r2, c2)`.
Subtract everything above row `r1` and everything left of column `c1`.
Their upper-left overlap was subtracted twice, so add `pre[r1][c1]` back once.
Keep all four signs and all four `+ 1` shifts exactly as written; a missing corner or wrong sign changes rectangles that do not begin at row `0` and column `0`.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    rows = int(tokens[0])
    columns = int(tokens[1])
    query_count = int(tokens[2])
    grid = []
    token_position = 3
    row = 0
    while row < rows:
        current_row = []
        column = 0
        while column < columns:
            current_row.append(int(tokens[token_position]))
            token_position = token_position + 1
            column = column + 1
        grid.append(current_row)
        row = row + 1

    pre = []
    row = 0
    while row < rows + 1:
        prefix_row = []
        column = 0
        while column < columns + 1:
            prefix_row.append(0)
            column = column + 1
        pre.append(prefix_row)
        row = row + 1

    row = 0
    while row < rows:
        column = 0
        while column < columns:
            pre[row + 1][column + 1] = grid[row][column] + pre[row][column + 1] + pre[row + 1][column] - pre[row][column]
            column = column + 1
        row = row + 1

    answer_text = ""
    query_index = 0
    while query_index < query_count:
        r1 = int(tokens[token_position])
        c1 = int(tokens[token_position + 1])
        r2 = int(tokens[token_position + 2])
        c2 = int(tokens[token_position + 3])
        rectangle_total = pre[r2 + 1][c2 + 1] - pre[r1][c2 + 1] - pre[r2 + 1][c1] + pre[r1][c1]
        if query_index > 0:
            answer_text = answer_text + "\n"
        answer_text = answer_text + str(rectangle_total)
        token_position = token_position + 4
        query_index = query_index + 1
    return answer_text

assert solve("2 3 2 2 1 4 3 5 6 0 1 1 2 1 2 1 2") == "16\n6"

## A Prefix-Sum Checklist

First, build the complete 1D prefix array or 2D prefix grid before answering any queries.
Second, translate every inclusive array or grid endpoint into the extra-border convention with `+ 1` on the ending index.
Third, test a single cell, the full array or grid, and a range that reaches the last position, row, and column.
A 2D prefix grid takes O(R · C) time to build, and each rectangle query takes O(1) time.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))